# Tutorial 3: Training a Personal Comfort Model

Learn how to use comfio's ML adapters to train a Personal Comfort Model (PCM)
with PyTorch.

**Requires**: `pip install comfio[torch]`

In [ ]:
import numpy as np
import pandas as pd
from comfio import SensorData, evaluate_thermal, calculate_global_ieq

## 1. Prepare IEQ Feature Data

In [ ]:
rng = np.random.default_rng(42)
n = 500

df = pd.DataFrame({
    'air_temp_c': rng.normal(24.0, 2.0, n),
    'radiant_temp_c': rng.normal(24.0, 1.5, n),
    'air_velocity_ms': rng.normal(0.1, 0.03, n),
    'relative_humidity_pct': rng.normal(50.0, 8.0, n),
    'illuminance_lux': rng.normal(500.0, 80.0, n),
    'noise_laeq_db': rng.normal(40.0, 7.0, n),
    'co2_ppm': rng.normal(800.0, 150.0, n),
})

sd = SensorData(df)
print(f'Loaded {len(df)} samples')

## 2. Extract IEQ Features with scikit-learn Transformer

In [ ]:
from comfio.ml.sklearn_transformers import IEQFeatureExtractor

extractor = IEQFeatureExtractor()
features = extractor.fit_transform(df)
print(f'Feature shape: {features.shape}')
print(f'Feature columns: {extractor.get_feature_names_out()}')

## 3. Create PyTorch Dataset

In [ ]:
from comfio.ml.torch_dataset import IEQTimeSeriesDataset

# Simulate comfort labels (TSV-like: -3 to +3)
labels = np.clip(np.round(rng.normal(0, 1, n)), -3, 3)

dataset = IEQTimeSeriesDataset(
    sensor_df=df,
    labels=labels,
    window_size=10,  # 10-timestep sliding window
    stride=5,
)
print(f'Dataset size: {len(dataset)}')
sample_x, sample_y = dataset[0]
print(f'Sample X shape: {sample_x.shape}, Y: {sample_y}')

## 4. Train a Simple PCM with PyTorch

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=16, shuffle=True)

model = nn.Sequential(
    nn.LSTM(input_size=7, hidden_size=32, batch_first=True),
)
head = nn.Linear(32, 1)

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(head.parameters()), lr=1e-3
)
criterion = nn.MSELoss()

for epoch in range(10):
    for batch_x, batch_y in loader:
        out, (hn, cn) = model(batch_x)
        pred = head(hn[-1]).squeeze(-1)
        loss = criterion(pred, batch_y.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/10 — Loss: {loss.item():.4f}')

## 5. Evaluate the Trained Model

In [ ]:
model.eval()
predictions = []
with torch.no_grad():
    for batch_x, _ in DataLoader(dataset, batch_size=32):
        out, (hn, _) = model(batch_x)
        pred = head(hn[-1]).squeeze(-1)
        predictions.extend(pred.numpy())

predictions = np.array(predictions)
print(f'Predictions range: [{predictions.min():.2f}, {predictions.max():.2f}]')
print(f'Mean prediction: {np.mean(predictions):.2f}')
print(f'Mean actual: {np.mean(labels[:len(predictions)]):.2f}')